# 8장. 작은 데이터 분석 프로젝트 완성하기

이 노트북은 **질문 → 원본 데이터 점검 → 전처리 → PK/FK 검증 → 안전한 병합 → completed 범위 집계 → 총합 대조 → 시각화 → 보고서 → 최종 Validation**을 하나의 재현 가능한 프로젝트로 연결합니다.

> 코드가 끝까지 실행되었다는 사실만으로 프로젝트가 검증된 것은 아닙니다. 같은 의미의 숫자와 공개 산출물이 정해 둔 기준을 함께 통과해야 합니다.


## 학습 목표

- 원본 데이터를 보존하며 전처리합니다.
- 주요 PK의 결측·중복과 FK 미매칭을 확인합니다.
- `validate`와 `indicator` 관점으로 병합 결과를 검증합니다.
- `line_total = quantity × unit_price`를 다시 확인합니다.
- 금액성 분석은 `order_status == "completed"` 범위를 사용합니다.
- 카테고리·월·고객 금액 합계를 같은 source total과 대조합니다.
- 날짜 오류로 월별 금액이 누락되지 않는지 확인합니다.
- 공개 고객 결과에서 원본 고객 ID와 직접 식별정보를 제거합니다.
- 결과 CSV, 그래프, Markdown 보고서와 PASS/FAIL Evidence를 저장합니다.
- 전체 파이프라인을 다시 실행해 재현성을 확인합니다.


## 제출 정보

- GitHub ID: `cyw0927`
- 작성일: 2026-09-17
- 최종 Notebook: `assignments/chapter08/chapter08.ipynb`
- 기준 Notebook: 강사 저장소 `notebooks/ch08_midterm_project.ipynb`
- 실습 가이드: `practice/chapter08/chapter08.md`


## 1. 프로젝트 루트와 출력 폴더 설정

VS Code에서 현재 작업 폴더가 프로젝트 루트 또는 하위 폴더일 수 있으므로 상위 폴더를 탐색합니다.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
FIGURE_DIR = REPORT_DIR / 'figures'

for path in [PROCESSED_DIR, REPORT_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python 실행 파일:', sys.executable)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('보고서 폴더:', REPORT_DIR)


## 2. 프로젝트 함수 불러오기

Notebook과 일괄 실행 스크립트가 같은 `src.midterm_project` 함수를 사용하도록 하여 계산 기준이 어긋나는 문제를 줄입니다.


In [ ]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    compare_shapes,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)
from src.midterm_project import (
    FORBIDDEN_CUSTOMER_COLUMNS,
    build_analysis_tables,
    build_interpretation_notes,
    build_key_duplicate_checks,
    build_midterm_report,
    build_project_validation,
    create_project_figures,
    run_midterm_project,
    save_project_tables,
    summarize_datasets,
)


## 3. 원본 데이터 확인

프로젝트는 `data/raw`의 원본 4개 파일에서 시작합니다.


In [ ]:
raw_data = load_sales_data(RAW_DIR)
dataset_summary = summarize_datasets(raw_data)
display(dataset_summary)

for name, df in raw_data.items():
    print(f'\n===== {name} =====')
    print('shape:', df.shape)
    print('columns:', df.columns.tolist())
    print('missing values:', int(df.isna().sum().sum()))
    print('duplicated rows:', int(df.duplicated().sum()))


## 4. 원본을 보존하며 전처리

`preprocess_sales_data()`는 원본 DataFrame을 직접 수정하지 않고 전처리된 복사본을 반환합니다. 전처리 전후 행 수가 달라졌다면 그 이유를 설명할 수 있어야 합니다.


In [ ]:
processed_data = preprocess_sales_data(raw_data)
processed_paths = save_processed_data(processed_data, PROCESSED_DIR)
preprocessing_comparison = compare_shapes(raw_data, processed_data)
display(preprocessing_comparison)

for path in processed_paths:
    print(path, 'OK' if path.exists() else 'MISSING')


## 5. PK와 FK를 먼저 검증

전체 행 중복과 PK 중복은 다릅니다. 고객·상품·주문·주문 상세의 PK는 **결측 0, 중복 0**이어야 합니다. FK는 부모 테이블에 없는 참조 ID가 0건이어야 합니다.


In [ ]:
key_duplicate_checks = build_key_duplicate_checks(processed_data)
relationship_checks = validate_relationships(processed_data).copy()

if not relationship_checks.empty:
    relationship_checks['status'] = (
        relationship_checks['invalid_count']
        .eq(0)
        .map({True: 'PASS', False: 'FAIL'})
    )

display(key_duplicate_checks)
display(relationship_checks)
assert key_duplicate_checks['status'].eq('PASS').all(), 'PK Validation 실패'
assert (not relationship_checks.empty and relationship_checks['invalid_count'].eq(0).all()), 'FK Validation 실패'
print('PK/FK Gate: PASS')


## 6. 안전한 병합과 계산 관계 검증

`build_analysis_tables()`는 병합 관계, 행 수, 미매칭, `line_total`, completed 범위, 날짜 오류를 함께 검증합니다.


In [ ]:
analysis_tables = build_analysis_tables(processed_data)
display(analysis_tables['merge_checks'])
display(analysis_tables['line_total_check'])
display(analysis_tables['date_checks'])
display(analysis_tables['amount_scope_summary'])
assert analysis_tables['merge_checks']['status'].eq('PASS').all()
assert analysis_tables['line_total_check']['status'].eq('PASS').all()
print('병합 / line_total Gate: PASS')


## 7. 핵심 분석 결과 확인

세 금액성 결과는 같은 `completed_order_sales`에서 출발합니다. 주문 상태 분포만 전체 주문을 사용합니다.


In [ ]:
category_sales = analysis_tables['category_sales']
monthly_sales = analysis_tables['monthly_sales']
customer_sales_internal = analysis_tables['customer_sales']
customer_sales_public = analysis_tables['customer_sales_public']
order_status_summary = analysis_tables['order_status_summary']
display(category_sales)
display(monthly_sales)
display(customer_sales_public.head(10))
display(order_status_summary)
print('내부 고객 결과 컬럼:', customer_sales_internal.columns.tolist())
print('공개 고객 결과 컬럼:', customer_sales_public.columns.tolist())


### 핵심 EDA 해석

#### 결과 1 — 카테고리별 completed 주문 기준 금액
기존 실습에서 스포츠 **31,743,000**, 전자기기 **26,400,000**으로 확인했다. 공식 프로젝트 실행 결과와 다시 대조한다.

- 관찰: 카테고리별 금액 차이가 크다.
- 판단: 금액 차이만으로 선호도 차이라고 단정하지 않는다.
- 의미: 판매 수량과 평균 단가를 분리하면 차이의 구조를 더 잘 볼 수 있다.
- 한계: 프로모션·재고·계절성 같은 원인 데이터가 없다.

#### 결과 2 — 월별 completed 주문 기준 금액과 주문 수
기존 실습에서 2025-07은 **5,869,000 / 8건**, 2025-08은 **15,621,000 / 18건**으로 확인했다. 공식 프로젝트 실행 결과와 다시 대조한다.

- 관찰: 7월보다 8월의 금액과 주문 수가 모두 높다.
- 판단: 증가 원인은 현재 데이터만으로 특정할 수 없다.
- 의미: 주문 수와 평균 주문 금액을 분리해서 추가 확인할 필요가 있다.
- 한계: 광고·할인·시즌 효과 같은 설명 변수는 없다.

#### 결과 3 — 고객별 completed 주문 구매 금액
고객별 구매 금액, 주문 횟수, 평균 주문 금액을 함께 확인한다.

- 관찰: 고객별 구매 규모에는 차이가 있다.
- 판단: 고액 구매 고객과 반복 구매 고객을 같은 의미로 보지 않는다.
- 의미: 고객 세분화나 반복 구매 분석으로 확장할 수 있다.
- 한계: 현재 결과만으로 충성도나 이탈 가능성을 판단할 수 없다.


## 8. 같은 의미의 금액 총합을 교차 검증

같은 completed 범위에서 만든 결과라면 completed source total = category total = monthly total = customer total이어야 합니다.


In [ ]:
total_consistency = analysis_tables['total_consistency_check']
display(total_consistency)
assert total_consistency['matches_completed'].all(), 'Total consistency Gate 실패'
assert analysis_tables['date_checks']['status'].eq('PASS').all(), '날짜 Gate 실패'
print('Total consistency / Date Gate: PASS')


## 9. 공개 고객 결과의 개인정보를 확인

내부 계산에는 `customer_id`가 필요할 수 있지만 공개 CSV·그래프·보고서에는 원본 고객 ID와 직접 식별정보를 포함하지 않습니다.


In [ ]:
forbidden = sorted(FORBIDDEN_CUSTOMER_COLUMNS.intersection(customer_sales_public.columns))
print('공개 결과 금지 컬럼:', forbidden)
assert not forbidden, 'Privacy Gate 실패'
assert 'customer_id' not in customer_sales_public.columns
print('Privacy Gate: PASS')


## 10. 최종 PASS/FAIL Validation

지금까지의 핵심 Gate를 하나의 표로 정리합니다. 하나라도 FAIL이면 완료 상태로 보지 않습니다.


In [ ]:
project_validation = build_project_validation(
    key_duplicate_checks,
    relationship_checks,
    analysis_tables,
)
display(project_validation)
assert project_validation['status'].eq('PASS').all(), '최종 Validation 실패'
print('Project Validation: PASS')


## 11. 해석 메모와 결과 CSV 저장


In [ ]:
interpretation_notes = build_interpretation_notes()
display(interpretation_notes)
saved_tables = save_project_tables(
    dataset_summary,
    preprocessing_comparison,
    key_duplicate_checks,
    relationship_checks,
    analysis_tables,
    interpretation_notes,
    REPORT_DIR,
)
for path in saved_tables:
    print(path.name, path.exists(), path.stat().st_size)


## 12. 저장된 공개 고객 CSV를 다시 검사


In [ ]:
saved_customer = pd.read_csv(REPORT_DIR / 'ch08_customer_sales.csv')
saved_forbidden = sorted(FORBIDDEN_CUSTOMER_COLUMNS.intersection(saved_customer.columns))
print('저장 CSV 컬럼:', saved_customer.columns.tolist())
print('금지 컬럼:', saved_forbidden)
assert not saved_forbidden
print('저장된 고객 CSV Privacy Gate: PASS')


## 13. 검증된 집계표에서 그래프 생성

그래프는 새로운 계산 범위를 만드는 단계가 아니라 검증된 숫자를 전달하는 단계입니다.


In [ ]:
saved_figures = create_project_figures(analysis_tables, FIGURE_DIR, show=True)
for path in saved_figures:
    print(path.name, path.exists(), path.stat().st_size)


### 대표 시각화 해석

- 카테고리별 completed 주문 기준 금액: 항목 간 크기 비교가 쉬워 막대그래프를 사용한다.
- 월별 completed 주문 기준 금액: 시간 흐름의 변화를 보기 위해 선그래프를 사용한다.
- 상위 익명 고객 구매 금액: 순위 비교를 위해 가로 막대그래프를 사용한다.

그래프는 앞에서 검증한 집계표를 사용하므로 원본 집계값과 일치하는지 함께 확인한다. 그래프만으로 프로모션 성공, 고객 선호, 수익성, 계절성의 원인을 단정하지 않는다.


## 14. Markdown 보고서 생성


In [ ]:
report_text = build_midterm_report(
    dataset_summary,
    preprocessing_comparison,
    key_duplicate_checks,
    relationship_checks,
    analysis_tables,
    interpretation_notes,
)
report_path = REPORT_DIR / 'ch08_midterm_report.md'
report_path.write_text(report_text, encoding='utf-8')
print('보고서 저장:', report_path)
print(report_text[:1800])


## 15. 전체 파이프라인 재실행

단계별 셀을 확인한 뒤 `run_midterm_project()`로 원본부터 보고서까지 다시 실행합니다.


In [ ]:
project_result = run_midterm_project(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    figure_dir=FIGURE_DIR,
    show_figures=False,
)
display(project_result['project_validation'])
project_result['report_path']


## 16. 최종 산출물 검증


In [ ]:
expected_outputs = [
    REPORT_DIR / 'ch08_midterm_report.md',
    REPORT_DIR / 'ch08_dataset_summary.csv',
    REPORT_DIR / 'ch08_preprocessing_comparison.csv',
    REPORT_DIR / 'ch08_key_duplicate_checks.csv',
    REPORT_DIR / 'ch08_relationship_checks.csv',
    REPORT_DIR / 'ch08_merge_checks.csv',
    REPORT_DIR / 'ch08_line_total_check.csv',
    REPORT_DIR / 'ch08_date_checks.csv',
    REPORT_DIR / 'ch08_amount_scope_summary.csv',
    REPORT_DIR / 'ch08_total_consistency_check.csv',
    REPORT_DIR / 'ch08_project_validation.csv',
    REPORT_DIR / 'ch08_category_sales.csv',
    REPORT_DIR / 'ch08_monthly_sales.csv',
    REPORT_DIR / 'ch08_customer_sales.csv',
    REPORT_DIR / 'ch08_order_status_summary.csv',
    REPORT_DIR / 'ch08_interpretation_notes.csv',
    FIGURE_DIR / 'ch08_category_sales.png',
    FIGURE_DIR / 'ch08_monthly_sales.png',
    FIGURE_DIR / 'ch08_top_customers.png',
]
all_outputs_valid = True
for path in expected_outputs:
    valid = path.exists() and path.stat().st_size > 0
    all_outputs_valid &= valid
    print(path.name, 'OK' if valid else 'MISSING OR EMPTY')
assert all_outputs_valid
print('전체 산출물 검증: PASS')


## 17. LLM 활용 기록

- 사용 여부: 예
- 사용 목적: 코드 의미 확인, 검증 항목 점검, 결과 해석 문장 정리
- Safe Context: 데이터 구조, 익명 집계 결과, 코드, 검증 결과
- Prompt 요약: Chapter 08 공식 Notebook과 실습 가이드 기준으로 누락된 검증 항목과 해석 구조를 확인
- 반영 방식: Notebook 출력과 검증 CSV로 확인 가능한 내용만 반영
- 검증 근거: `ch08_total_consistency_check.csv`, `ch08_project_validation.csv`, Notebook Output을 서로 대조

원본 고객 행이나 직접 식별정보는 LLM 입력으로 사용하지 않는다.


## 18. 실습 과제 정리

### 분석 질문
1. 카테고리별 completed 주문 기준 금액은 어떻게 다른가?
2. 월별 completed 주문 기준 금액과 주문 수는 어떻게 변하는가?
3. completed 주문 기준 고객별 구매 금액에는 어떤 차이가 있는가?

### 핵심 인사이트
1. 카테고리별 completed 주문 기준 금액에는 뚜렷한 차이가 있다.
2. 월별 completed 주문 금액과 주문 수는 기간별로 차이가 있으며, 금액 변화는 주문 수와 평균 주문 금액을 분리해서 봐야 한다.
3. 고객별 구매 규모는 차이가 있으므로 총구매금액만이 아니라 주문 횟수와 평균 주문 금액을 함께 확인하는 것이 필요하다.

### 가장 중요한 분석적 의미
큰 숫자를 찾는 것보다 동일한 분석 범위와 계산 기준을 유지하고, 서로 다른 집계 결과의 총합을 교차 검증하는 과정이 중요하다.

### 한계
- 프로모션, 광고, 재고, 계절성 데이터가 없다.
- completed 주문 기준 금액을 회계상 순매출이라고 단정할 수 없다.
- 집계 결과만으로 고객 행동의 원인을 설명할 수 없다.

### 다음 분석 제안
1. 카테고리별 판매 수량과 평균 단가 분리 비교
2. 월별 주문 수와 평균 주문 금액 분리 비교
3. 고객별 최근 구매일·구매 빈도·평균 구매 금액 추가 분석

### 최종 제출 전 확인
- `project_validation`의 모든 항목이 PASS인지 확인
- `total_consistency_check`가 모두 일치하는지 확인
- completed 주문의 날짜 오류가 0건인지 확인
- 공개 고객 CSV에 직접 식별정보가 없는지 확인
- `python scripts/run_midterm_project.py` 재실행 결과와 Notebook 핵심 수치가 일치하는지 확인


## 정리

이번 장에서는 원본 데이터 확인, 전처리, PK/FK, 병합, `line_total`, completed 범위, total consistency, 날짜, 개인정보, 시각화, 보고서, 최종 Validation을 하나의 프로젝트로 연결했습니다.

```text
실행 성공 ≠ 검증 완료
같은 의미의 숫자는 서로 다른 집계에서도 같아야 한다
공개 결과에는 불필요한 개인정보를 남기지 않는다
프로젝트는 원본부터 다시 실행할 수 있어야 한다
```
